# 02 — Chain-, Tree- and Graph-of-Thoughts

What CoT, ToT and GoT actually do in Athena, and what was (and was not) measured. Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


In [3]:
pd.DataFrame([status_row(m) for m in ("cot", "tot", "got", "self-consistency")]).T

,0,1,2,3
method,Chain-of-Thought (Chain Aggregation),Tree-of-Thoughts (Tree Aggregation),Graph-of-Thoughts (Graph Consensus) + preprocessing GoT trace,Self-Consistency
implemented,IMPLEMENTED,IMPLEMENTED,IMPLEMENTED,NOT_IMPLEMENTED
wired,WIRED,WIRED,WIRED,NOT_WIRED
authority,CONFIDENCE_ONLY,CONFIDENCE_ONLY,CONFIDENCE_ONLY,NONE
evaluation,SIMULATED (mock runner),SIMULATED (mock runner),SIMULATED (mock runner),NOT_RUN
production,API_ONLY,API_ONLY,API_ONLY,NOT_IN_PRODUCT
recommendation,RESEARCH,RESEARCH,RESEARCH,RESEARCH
result,NOT_RUN with a real provider in this repo; the mock shows ≈3× calls/tokens for chain/tree/graph *prompting* variants — not this aggregat...,NOT_RUN with a real provider in this repo.,NOT_RUN with a real provider in this repo.,UNKNOWN
limitations,"Training Wizard stored council_config.reasoning_mode and kind='voting' with the run, but the training path never runs the voting council...","Training Wizard stored council_config.reasoning_mode and kind='voting' with the run, but the training path never runs the voting council...","Training Wizard stored council_config.reasoning_mode and kind='voting' with the run, but the training path never runs the voting council...","Closest relative in Athena is the council's majority vote over different personas, which is not the same method (different prompts, not ..."


## What the wiring is

In Athena, *CoT / ToT / GoT* are **post-hoc aggregations over council member answers**, not reasoning performed by a model. Each member is called once; the chain, tree or graph is built from the returned text and makes **zero** model calls of its own. Its only effect is a confidence boost (×1.15) for the member it prefers. Under the default `vote` strategy that boost only breaks ties; under `confidence`, `consensus` and `debate` it can change the winner. These methods are therefore **confidence / tie-break behaviour**, not evidence of chain-of-thought, tree search or graph reasoning by a language model.

In [4]:
print(REG["cot"]["affects_final"])
print()
print(REG["cot"]["limitations"])

Boosts the confidence of the member it prefers by ×1.15 before voting. Under the default 'vote' strategy the winner is the largest answer group (confidence only breaks ties and picks the answer text); under 'confidence', 'consensus' and 'debate' the boost can change the winner.

Training Wizard stored council_config.reasoning_mode and kind='voting' with the run, but the training path never runs the voting council: only kind='proposer_judge' is consumed (model_training._maybe_council_override) and use_got only adds a preprocessing trace graph. Since 2026-11-14 the standard wizard offers Proposer / Judge only; restored voting configs are kept and labelled research only. Not iterative CoT prompting; raw chain-of-thought is never stored.


## The only frozen evidence — SIMULATED

A reasoning-strategies harness exists and was run with a **mock runner** (6 configurations × 5 tasks). A mock cannot measure reasoning quality; it measures call and token overhead of the prompting variants and that the defect-injection checks fire.

In [5]:
print((EV / "reasoning_methods/reasoning_strategies_mock_benchmark.md").read_text())

<!-- Excerpt of the frozen offline mock-harness report. Mock runner; no real provider call. -->
Benchmark (llm/reasoning_benchmark.py, 6 configurations × 5 representative tasks):  NOT_RUN for a real provider
Mock runner (offline, deterministic answers from the packet, persona disagreement injected):
  configuration        calls  tokens   grounded  unsupported  directional  coverage  agreement
  deterministic_only     0        0      5/5         0            0          1.0        —
  single_llm             5     3 540     5/5         0            0          1.0        —
  council_vote          15    10 646     5/5         0            0          1.0       0.667
  chain_aggregation     15    10 646     5/5         0            0          1.0       0.667
  tree_aggregation      15    10 646     5/5         0            0          1.0       0.667
  graph_consensus       15    10 646     5/5         0            0          1.0       0.667
Defect injection: wrong_direction → 3 directional vi

**RESULT.** `NOT_RUN` with a real provider. No accuracy, cost or latency claim about CoT, ToT or GoT is made. Self-consistency (k samples, majority) is `NOT_IMPLEMENTED`; the council's vote over different personas is a different method. The research plan for a real measurement is in `METHODOLOGY.md` (roadmap state `RESEARCH_PLANNED`).